In [12]:
import os
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Model storage location
BASE = os.path.join(os.path.dirname(os.getcwd()), "Task2", "models", "model_variants")

print(BASE)
print(os.path.exists(BASE))
# Existing WER measurements
wer_scores = {
    "whisper-audio-to-text": 0.14775725593667546,
    "s2t-small-librispeech-asr": 0.9285270836566817,
    "whisper-tiny": 0.19882042526773241
}

# Calculate folder size in MB
def get_folder_size(path):
    if not os.path.exists(path):
        return None

    total = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path)
        for f in files
    )

    return round(total / (1024 ** 2), 2)


# Calculate individual model sizes
print("MODEL SIZES")
print("=" * 40)
asr_sizes = {}
sentiment_sizes = {}

for folder in os.listdir(os.path.join(BASE, "asr")):
    path = os.path.join(BASE, "asr", folder)
    if os.path.isdir(path):
        asr_sizes[folder] = get_folder_size(path)
        print(f"ASR - {folder}: {asr_sizes[folder]} MB")

for folder in os.listdir(os.path.join(BASE, "sentiment")):
    path = os.path.join(BASE, "sentiment", folder)
    if os.path.isdir(path):
        sentiment_sizes[folder] = get_folder_size(path)
        print(f"Sentiment - {folder}: {sentiment_sizes[folder]} MB")



# Combine evaluation reports
all_reports = []

for file in glob.glob("report*.csv"):

    df = pd.read_csv(file, index_col=0)
    df.columns = [str(c).strip().lower() for c in df.columns]

    model = os.path.basename(file).replace(
        "report_", ""
    ).replace(".csv", "")

    # Split ASR and sentiment names
    asr_model, sentiment_model = model.split("__", 1)

    row = {
        "model": model,
        "asr_model": asr_model,
        "sentiment_model": sentiment_model,
        "version": "1.0",
        "source": "Hugging Face",

        # Individual model sizes
        "asr_size_mb": asr_sizes.get(asr_model),
        "sentiment_size_mb": sentiment_sizes.get(sentiment_model),

        # Total pipeline size
        "total_size_mb": asr_sizes[asr_model] + sentiment_sizes[sentiment_model],

        # ASR evaluation
        "WER": wer_scores.get(asr_model)
    }

    # Extract sentiment evaluation metrics
    for label, r in df.iterrows():

        label = str(label).strip().lower()

        if label in ["negative", "neutral", "positive"]:

            for metric in ["precision", "recall", "f1-score", "support"]:
                if metric in df.columns:
                    row[f"{label}_{metric}"] = r[metric]

        elif label == "accuracy":
            row["accuracy"] = r["f1-score"]

        elif label == "macro avg":
            row["macro_precision"] = r["precision"]
            row["macro_recall"] = r["recall"]
            row["macro_f1"] = r["f1-score"]

        elif label == "weighted avg":
            row["weighted_f1"] = r["f1-score"]

    all_reports.append(row)


# Save final metadata
metadata = pd.DataFrame(all_reports)

metadata.to_csv("report_combined.csv", index=False)

print("\nFINAL METADATA")
print("=" * 40)
print(metadata.to_string(index=False))

print(f"\nCombined {len(all_reports)} pipeline reports.")
print("Saved to report_combined.csv")

C:\Users\nova\Desktop\Milestone2\Task2\models\model_variants
True
MODEL SIZES
ASR - s2t-small-librispeech-asr: 348.97 MB
ASR - whisper-audio-to-text: 140.32 MB
ASR - whisper-tiny: 580.65 MB
Sentiment - distilbert-base-uncased-finetuned-sst-2-english: 1278.59 MB
Sentiment - multilingual-sentiment-analysis: 520.0 MB
Sentiment - sentiment-roberta-large-english: 4072.68 MB
Sentiment - twitter-roberta-base-sentiment-latest: 954.87 MB

FINAL METADATA
                                                                     model                 asr_model                                 sentiment_model version       source  asr_size_mb  sentiment_size_mb  total_size_mb      WER  negative_precision  negative_recall  negative_f1-score  negative_support  neutral_precision  neutral_recall  neutral_f1-score  neutral_support  positive_precision  positive_recall  positive_f1-score  positive_support  accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
s2t-small-librispeech-asr__distilbert-base-

In [22]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def baseline_sentiment_model(dataframe: pd.DataFrame):
    # Always predict neutral
    return ["Neutral"] * len(dataframe)


# Run baseline on the test slice
test_slice = pd.read_csv("test.csv")
test_slice["baseline_prediction"] = baseline_sentiment_model(test_slice)

print("\nBASELINE MODEL (always predict neutral)")
print("========================================")

baseline = classification_report(
    test_slice["label"],
    test_slice["baseline_prediction"],
    labels=["Negative", "Neutral", "Positive"],
    output_dict=True,
    zero_division=0
)


BASELINE MODEL (always predict neutral)


In [24]:
baseline_row = {
    "model": "BASELINE_always_neutral",
    "accuracy": baseline["accuracy"],

    "macro_precision": baseline["macro avg"]["precision"],
    "macro_recall": baseline["macro avg"]["recall"],
    "macro_f1": baseline["macro avg"]["f1-score"],

    "weighted_f1": baseline["weighted avg"]["f1-score"]
}

# Add per-class metrics
for label in ["Negative", "Neutral", "Positive"]:
    for metric in ["precision", "recall", "f1-score", "support"]:
        baseline_row[f"{label.lower()}_{metric}"] = baseline[label][metric]

# Add to existing metadata
metadata = pd.concat(
    [metadata, pd.DataFrame([baseline_row])],
    ignore_index=True
)

metadata.to_csv("report_combined.csv", index=False)

print("Baseline added successfully!")


Baseline added successfully!
